# 构建文本生成应用

通过本课程你已经了解到，有核心概念如提示词(prompt)甚至还有一整门学科叫做“提示工程”。你可以使用的许多工具，如 ChatGPT、Office 365、Microsoft Power Platform 等，都支持通过提示词来完成某些任务。

如果你想为应用添加这样的体验，你需要了解提示词、补全等概念，并选择一个库来使用。这正是你将在本章中学习的内容。

## 介绍

在本章中，你将：

- 了解 openai 库及其核心概念。
- 使用 openai 构建一个文本生成应用。
- 理解如何使用提示词、温度和令牌等概念来构建文本生成应用。

## 学习目标

本课结束时，你将能：

- 解释什么是文本生成应用。
- 使用 openai 构建文本生成应用。
- 配置应用使用更多或更少的令牌，同时改变温度，以获得多样化输出。

## 什么是文本生成应用？

通常你构建的应用都有某种界面，例如：

- 基于命令。控制台应用是典型的基于命令的应用，你输入命令，应用执行任务。例如，`git` 就是一个基于命令的应用。
- 用户界面(UI)。一些应用有图形用户界面(GUI)，你可以点击按钮、输入文本、选择选项等。

### 控制台和 UI 应用的局限性

与基于命令的应用相比，你只能输入命令：

- <strong>它是有限的</strong>。你不能随意输入任何命令，只能输入应用支持的命令。
- <strong>语言特定</strong>。一些应用支持多种语言，但默认是为特定语言构建的，即使你能添加更多语言支持。

### 文本生成应用的优势

那么文本生成应用有什么不同呢？

文本生成应用更灵活，不受限于特定命令集或输入语言。相反，你可以使用自然语言与应用交互。另一个好处是，你已经在与经过大量信息训练的数据源交互，而传统应用可能受限于数据库中的内容。

### 用文本生成应用我能构建什么？

你可以构建多种应用。例如：

- <strong>聊天机器人</strong>。一个能回答关于你公司及产品问题的聊天机器人是很好的例子。
- <strong>助手</strong>。大型语言模型（LLM）擅长总结文本、获取文本洞察、生成简历等。
- <strong>代码助手</strong>。根据所用语言模型不同，你可以构建代码助手帮你写代码。例如，你可以使用 GitHub Copilot 或 ChatGPT 帮助写代码。

## 怎么开始？

你需要找到集成大型语言模型（LLM）的方法，通常有两个途径：

- 使用 API。你通过构建网络请求发送提示词，获取生成文本。
- 使用库。库封装了 API 调用，使使用更简单。

## 库/SDK

有几个知名的 LLM 库，例如：

- **openai**，这个库让你轻松连接到模型并发送提示词。

还有一些更高级的库，比如：

- **Langchain**。Langchain 很著名，支持 Python。
- **Semantic Kernel**。Semantic Kernel 是微软的库，支持 C#、Python 和 Java。

## 使用 openai 构建第一个应用

让我们看看如何构建第一个应用，需要哪些库，要求是什么。

### 安装 openai

  > [!NOTE] 如果你在 Codespaces 或 Devcontainer 中运行本笔记本，这一步不必执行


有很多库可以与 OpenAI 或 Azure OpenAI 交互，也支持多种编程语言，如 C#、Python、JavaScript、Java 等。  
我们选择使用 `openai` Python 库，因此用 `pip` 安装它。

```bash
pip install openai
```

若不在 Codespaces 或 Dev Container 中运行本笔记本，还需在本机安装 [Python](https://www.python.org/)。

### 创建资源

如果尚未完成，请执行以下步骤：

- 在 Azure 注册账户 <https://azure.microsoft.com/free/>。
- 在 [Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) 创建 Azure OpenAI 资源。有关如何[创建资源](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst)的指南。


### 找到 API 密钥和端点

现在你需要告诉 `openai` 库使用哪个 API 密钥。去 Azure OpenAI 资源的“密钥和端点”部分，复制“密钥1”的值。

  ![Azure Portal 中的密钥和端点资源面板](https://learn.microsoft.com/azure/ai-foundry/openai/media/quickstarts/endpoint.png?WT.mc_id=academic-105485-koreyst)

复制后，接下来我们告诉库使用这些信息。

> [!NOTE]
> 将 API 密钥与代码分开是值得的。你可以使用环境变量实现。
> - 在 .env 文件中设置环境变量 `AZURE_OPENAI_API_KEY` 为你的 API 密钥。如果你已完成本课程之前的练习，配置已完成。


### 设置 Azure 配置

如果使用 Azure OpenAI，配置方法如下。Responses API 从 Azure OpenAI **v1 端点** 提供服务，因此我们将 `OpenAI` 客户端指向 `<your-endpoint>/openai/v1/`：

```python
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment = os.environ['AZURE_OPENAI_DEPLOYMENT']
```

上述代码设置了：

- `api_key`，这是你在 Azure 门户找到的 API 密钥。
- `base_url`，这是你 Azure OpenAI 端点，后面加了 `/openai/v1/`。端点在 Azure 门户中 API 密钥旁边找到。使用 v1 端点意味着你无需再传递 `api_version`。
- `deployment`，这是你在 Foundry 门户创建的模型部署名称。

> [!NOTE]
> `os.environ` 是读取环境变量的映射。你可以用它读取如 `AZURE_OPENAI_API_KEY` 和 `AZURE_OPENAI_ENDPOINT` 的环境变量。

## 生成文本

生成文本的方法是使用 `responses.create`。示例：

```python
prompt = "Complete the following: Once upon a time there was a"

response = client.responses.create(model=deployment, input=prompt, store=False)
print(response.output_text)
```

上面代码中，我们创建了响应对象，传入想用的模型和提示词，然后打印生成的文本。

### 聊天响应

Responses API 适合单轮文本生成和多轮聊天机器人——只需在 `input` 列表中提供更多消息构建对话：

```python
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment = os.environ['AZURE_OPENAI_DEPLOYMENT']

response = client.responses.create(model=deployment, input=[{"role": "user", "content": "Hello world"}], store=False)
print(response.output_text)
```

更多关于此功能将在后续章节讲解。

## 练习 - 你的第一个文本生成应用

现在我们已经学习如何设置和配置 Azure OpenAI 服务，是时候构建你的第一个文本生成应用了。按以下步骤操作：


1. 创建一个虚拟环境并安装 openai：

  > [!NOTE] 如果你在 Codespaces 或 Devcontainer 中运行此笔记本，则此步骤不是必需的


In [ ]:
# Create virtual environment
! python -m venv venv
# Activate virtual environment
! source venv/bin/activate
# Install openai package
! pip install openai

> [!NOTE]
> 如果你使用的是 Windows，请输入 `venv\Scripts\activate` 而不是 `source venv/bin/activate`。

> [!NOTE]
> 通过访问 https://portal.azure.com/ 并搜索 `Open AI`，选择 `Open AI 资源`，然后选择 `密钥和终结点`，复制 `密钥 1` 的值，即可找到你的 Azure OpenAI 密钥。


1. 创建一个 *app.py* 文件，并输入以下代码：


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']

# add your completion code
prompt = "Complete the following: Once upon a time there was a"

# make a request using the Responses API
response = client.responses.create(model=deployment, input=prompt, store=False)

# print response
print(response.output_text)



    你应该看到如下输出：

    ```output
     非常不开心的 _____。

    从前有一只非常不开心的人鱼。
    ```


## 不同类型的提示，用于不同的事情

现在你已经看到了如何使用提示生成文本。你甚至有一个已经运行的程序，可以修改和更改它以生成不同类型的文本。

提示可以用于各种任务。例如：

- <strong>生成某种类型的文本</strong>。例如，你可以生成一首诗、测验问题等。
- <strong>查询信息</strong>。你可以使用提示查找信息，比如下面的例子“CORS在Web开发中是什么意思？”。
- <strong>生成代码</strong>。你可以使用提示生成代码，例如开发用于验证电子邮件的正则表达式，或者为什么不生成一个完整的程序，比如一个Web应用？  

## 一个更实用的用例：食谱生成器

想象你家里有食材，想做点什么。为此，你需要一个食谱。找到食谱的方法之一是使用搜索引擎，或者你也可以使用大型语言模型来完成。

你可以这样写一个提示：

> “给我展示5个使用以下食材做成的菜谱：鸡肉、土豆和胡萝卜。每个菜谱列出所有使用的食材”

根据上述提示，你可能会得到类似的回答：

```output
1. Roasted Chicken and Vegetables: 
Ingredients: 
- 4 chicken thighs
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 2 tablespoons olive oil
- 2 cloves garlic, minced
- 1 teaspoon dried thyme
- 1 teaspoon dried oregano
- Salt and pepper, to taste

2. Chicken and Potato Stew: 
Ingredients: 
- 2 tablespoons olive oil
- 1 onion, diced
- 2 cloves garlic, minced
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 1 cup chicken broth
- Salt and pepper, to taste

3. Chicken and Potato Bake: 
Ingredients: 
- 2 tablespoons olive oil
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 onion, diced
- 2 cloves garlic, minced
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 1 cup chicken broth
- Salt and pepper, to taste

4. Chicken and Potato Soup: 
Ingredients: 
- 2 tablespoons olive oil
- 1 onion, diced
- 2 cloves garlic, minced
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 4 cups chicken broth
- Salt and pepper, to taste

5. Chicken and Potato Hash: 
Ingredients: 
- 2 tablespoons olive oil
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 onion, diced
- 2 cloves garlic, minced
- 1 teaspoon dried oregano
```

这个结果很好，我知道该做什么了。此时，有用的改进可能是：

- 过滤掉我不喜欢或过敏的食材。
- 生成购物清单，以防我家里没有所有食材。

针对上述情况，我们添加一个附加提示：

> “请去除含有大蒜的食谱，因为我过敏，并用其他东西替代。同时，请根据我家里已有的鸡肉、土豆和胡萝卜，为这些菜谱生成购物清单。”

现在你会得到新的结果，也就是：

```output
1. Roasted Chicken and Vegetables: 
Ingredients: 
- 4 chicken thighs
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 2 tablespoons olive oil
- 1 teaspoon dried thyme
- 1 teaspoon dried oregano
- Salt and pepper, to taste

2. Chicken and Potato Stew: 
Ingredients: 
- 2 tablespoons olive oil
- 1 onion, diced
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 1 cup chicken broth
- Salt and pepper, to taste

3. Chicken and Potato Bake: 
Ingredients: 
- 2 tablespoons olive oil
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 onion, diced
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 1 cup chicken broth
- Salt and pepper, to taste

4. Chicken and Potato Soup: 
Ingredients: 
- 2 tablespoons olive oil
- 1 onion, diced
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 teaspoon dried oregano
- 1 teaspoon dried thyme
- 4 cups chicken broth
- Salt and pepper, to taste

5. Chicken and Potato Hash: 
Ingredients: 
- 2 tablespoons olive oil
- 2 chicken breasts, cut into cubes
- 2 potatoes, cut into cubes
- 2 carrots, cut into cubes
- 1 onion, diced
- 1 teaspoon dried oregano

Shopping List: 
- Olive oil
- Onion
- Thyme
- Oregano
- Salt
- Pepper
```

这就是你的五个食谱，里面没有提到大蒜，还根据你已有的食材生成了购物清单。


## 练习 - 构建一个食谱生成器

现在我们已经演示了一个场景，让我们编写代码来匹配所演示的场景。为此，请执行以下步骤：

1. 使用现有的 *app.py* 文件作为起点
1. 找到 `prompt` 变量并将其代码更改为以下内容：


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# load environment variables from .env file
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment = os.environ['AZURE_OPENAI_DEPLOYMENT']

prompt = "Show me 5 recipes for a dish with the following ingredients: chicken, potatoes, and carrots. Per recipe, list all the ingredients used"

# make a request using the Responses API
response = client.responses.create(model=deployment, input=prompt, max_output_tokens=600, store=False)

# print response
print(response.output_text)


如果你现在运行代码，你应该会看到类似的输出：

```output
-Chicken Stew with Potatoes and Carrots: 3 tablespoons oil, 1 onion, chopped, 2 cloves garlic, minced, 1 carrot, peeled and chopped, 1 potato, peeled and chopped, 1 bay leaf, 1 thyme sprig, 1/2 teaspoon salt, 1/4 teaspoon black pepper, 1 1/2 cups chicken broth, 1/2 cup dry white wine, 2 tablespoons chopped fresh parsley, 2 tablespoons unsalted butter, 1 1/2 pounds boneless, skinless chicken thighs, cut into 1-inch pieces
-Oven-Roasted Chicken with Potatoes and Carrots: 3 tablespoons extra-virgin olive oil, 1 tablespoon Dijon mustard, 1 tablespoon chopped fresh rosemary, 1 tablespoon chopped fresh thyme, 4 cloves garlic, minced, 1 1/2 pounds small red potatoes, quartered, 1 1/2 pounds carrots, quartered lengthwise, 1/2 teaspoon salt, 1/4 teaspoon black pepper, 1 (4-pound) whole chicken
-Chicken, Potato, and Carrot Casserole: cooking spray, 1 large onion, chopped, 2 cloves garlic, minced, 1 carrot, peeled and shredded, 1 potato, peeled and shredded, 1/2 teaspoon dried thyme leaves, 1/4 teaspoon salt, 1/4 teaspoon black pepper, 2 cups fat-free, low-sodium chicken broth, 1 cup frozen peas, 1/4 cup all-purpose flour, 1 cup 2% reduced-fat milk, 1/4 cup grated Parmesan cheese

-One Pot Chicken and Potato Dinner: 2 tablespoons olive oil, 1 pound boneless, skinless chicken thighs, cut into 1-inch pieces, 1 large onion, chopped, 3 cloves garlic, minced, 1 carrot, peeled and chopped, 1 potato, peeled and chopped, 1 bay leaf, 1 thyme sprig, 1/2 teaspoon salt, 1/4 teaspoon black pepper, 2 cups chicken broth, 1/2 cup dry white wine

-Chicken, Potato, and Carrot Curry: 1 tablespoon vegetable oil, 1 large onion, chopped, 2 cloves garlic, minced, 1 carrot, peeled and chopped, 1 potato, peeled and chopped, 1 teaspoon ground coriander, 1 teaspoon ground cumin, 1/2 teaspoon ground turmeric, 1/2 teaspoon ground ginger, 1/4 teaspoon cayenne pepper, 2 cups chicken broth, 1/2 cup dry white wine, 1 (15-ounce) can chickpeas, drained and rinsed, 1/2 cup raisins, 1/2 cup chopped fresh cilantro
```

> NOTE，你的LLM是非确定性的，因此每次运行程序时可能会得到不同的结果。

太好了，让我们看看如何改进。为了改进，我们希望确保代码具有灵活性，以便成分和食谱数量可以改进和更改。


1. 让我们用以下方式更改代码：


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# load environment variables from .env file
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment = os.environ['AZURE_OPENAI_DEPLOYMENT']

no_recipes = input("No of recipes (for example, 5: ")

ingredients = input("List of ingredients (for example, chicken, potatoes, and carrots: ")

# interpolate the number of recipes into the prompt an ingredients
prompt = f"Show me {no_recipes} recipes for a dish with the following ingredients: {ingredients}. Per recipe, list all the ingredients used"

# make a request using the Responses API
response = client.responses.create(model=deployment, input=prompt, max_output_tokens=600, store=False)

# print response
print(response.output_text)


-草莓奶油蛋糕：牛奶，面粉，泡打粉，糖，盐，无盐黄油，草莓，鲜奶油        
-草莓牛奶：牛奶，草莓，糖，香草提取物
```

### Improve by adding filter and shopping list

We now have a working app capable of producing recipes and it's flexible as it relies on inputs from the user, both on the number of recipes but also the ingredients used.

To further improve it, we want to add the following:

- **Filter out ingredients**. We want to be able to filter out ingredients we don't like or are allergic to. To accomplish this change, we can edit our existing prompt and add a filter condition to the end of it like so:

    ```python
    filter = input("Filter (for example, vegetarian, vegan, or gluten-free: ")

    prompt = f"Show me {no_recipes} recipes for a dish with the following ingredients: {ingredients}. Per recipe, list all the ingredients used, no {filter}"
    ```

    Above, we add `{filter}` to the end of the prompt and we also capture the filter value from the user.

    An example input of running the program can now look like so:
    
    ```output    
    No of recipes (for example, 5: 3
    List of ingredients (for example, chicken, potatoes, and carrots: onion,milk
    Filter (for example, vegetarian, vegan, or gluten-free: no milk

    1. French Onion Soup

    Ingredients:
    
    -1 large onion, sliced
    -3 cups beef broth
    -1 cup milk
    -6 slices french bread
    -1/4 cup shredded Parmesan cheese
    -1 tablespoon butter
    -1 teaspoon dried thyme
    -1/4 teaspoon salt
    -1/4 teaspoon black pepper
    
    Instructions:
    
    1. In a large pot, sauté onions in butter until golden brown.
    2. Add beef broth, milk, thyme, salt, and pepper. Bring to a boil.
    3. Reduce heat and simmer for 10 minutes.
    4. Place french bread slices on soup bowls.
    5. Ladle soup over bread.
    6. Sprinkle with Parmesan cheese.
    
    2. Onion and Potato Soup
    
    Ingredients:
    
    -1 large onion, chopped
    -2 cups potatoes, diced
    -3 cups vegetable broth
    -1 cup milk
    -1/4 teaspoon black pepper
    
    Instructions:
    
    1. In a large pot, sauté onions in butter until golden brown.
    2. Add potatoes, vegetable broth, milk, and pepper. Bring to a boil.
    3. Reduce heat and simmer for 10 minutes.
    4. Serve hot.
    
    3. Creamy Onion Soup
    
    Ingredients:
    
    -1 large onion, chopped
    -3 cups vegetable broth
    -1 cup milk
    -1/4 teaspoon black pepper
    -1/4 cup all-purpose flour
    -1/2 cup shredded Parmesan cheese
    
    Instructions:
    
    1. In a large pot, sauté onions in butter until golden brown.
    2. Add vegetable broth, milk, and pepper. Bring to a boil.
    3. Reduce heat and simmer for 10 minutes.
    4. In a small bowl, whisk together flour and Parmesan cheese until smooth.
    5. Add to soup and simmer for an additional 5 minutes, or until soup has thickened.
    ```

    As you can see, any recipes with milk in it has been filtered out. But, if you're lactose intolerant, you might want to filter out recipes with cheese in them as well, so there's a need to be clear.

- **Produce a shopping list**. We want to produce a shopping list, considering what we already have at home.

    For this functionality, we could either try to solve everything in one prompt or we could split it up into two prompts. Let's try the latter approach. Here we're suggesting adding an additional prompt, but for that to work, we need to add the result of the former prompt as context to the latter prompt. 

    Locate the part in the code that prints out the result from the first prompt and add the following code below:
    
    ```python
    old_prompt_result = response.output_text
    prompt = "Produce a shopping list for the generated recipes and please don't include ingredients that I already have."
    
    new_prompt = f"{old_prompt_result} {prompt}"
    messages = [{"role": "user", "content": new_prompt}]
    response = client.responses.create(model=deployment, input=messages, max_output_tokens=1200, store=False)
    
    # print response
    print("Shopping list:")
    print(response.output_text)
    ```

    Note the following:

    - We're constructing a new prompt by adding the result from the first prompt to the new prompt: 
    
        ```python
        new_prompt = f"{old_prompt_result} {prompt}"
        messages = [{"role": "user", "content": new_prompt}]
        ```

    - We make a new request, but also considering the number of tokens we asked for in the first prompt, so this time we say `max_output_tokens` is 1200. 

        ```python
        response = client.responses.create(model=deployment, input=messages, max_output_tokens=1200, store=False)
        ```  

        Taking this code for a spin, we now arrive at the following output:

        ```output
        No of recipes (for example, 5: 2
        List of ingredients (for example, chicken, potatoes, and carrots: apple,flour
        Filter (for example, vegetarian, vegan, or gluten-free: sugar
        Recipes:
         or milk.
        
        -Apple and flour pancakes: 1 cup flour, 1/2 tsp baking powder, 1/2 tsp baking soda, 1/4 tsp salt, 1 tbsp sugar, 1 egg, 1 cup buttermilk or sour milk, 1/4 cup melted butter, 1 Granny Smith apple, peeled and grated
        -Apple fritters: 1-1/2 cups flour, 1 tsp baking powder, 1/4 tsp salt, 1/4 tsp baking soda, 1/4 tsp nutmeg, 1/4 tsp cinnamon, 1/4 tsp allspice, 1/4 cup sugar, 1/4 cup vegetable shortening, 1/4 cup milk, 1 egg, 2 cups shredded, peeled apples
        Shopping list:
         -Flour, baking powder, baking soda, salt, sugar, egg, buttermilk, butter, apple, nutmeg, cinnamon, allspice 
        ```
        
- **A word on token length**. We should consider how many tokens we need to generate the text we want. Tokens cost money, so where possible, we should try to be economical with the number of tokens we use. For example, can we phrase the prompt so that we can use less tokens?

   To change tokens used, you can use the `max_output_tokens` parameter. For example, if you want to use 100 tokens, you would do:

    ```python
    response = client.responses.create(model=deployment, input=messages, max_output_tokens=100, store=False)
    ```

- **Experimenting with temperature**. Temperature is something we haven't mentioned so far but is an important context for how our program performs. The higher the temperature value the more random the output will be. Conversely the lower the temperature value the more predictable the output will be. Consider whether you want variation in your output or not.

   To alter the temperature, you can use the `temperature` parameter. For example, if you want to use a temperature of 0.5, you would do:

    ```python
    response = client.responses.create(model=deployment, input=messages, store=False)
    ```

   > Note, the closer to 1.0, the more varied the output.



## Assignment

For this assignment, you can choose what to build.

Here are some suggestions:

- Tweak the recipe generator app to improve it further. Play around with temperature values, and the prompts to see what you can come up with.
- Build a "study buddy". This app should be able to answer questions about a topic for example Python, you could have prompts like "What is a certain topic in Python?", or you could have a prompt that says, show me code for a certain topic etc.
- History bot, make history come alive, instruct the bot to play a certain historical character and ask it questions about its life and times. 

## Solution

### Study buddy

- "You're an expert on the Python language

    Suggest a beginner lesson for Python in the following format:
    
    Format:
    - concepts:
    - brief explanation of the lesson:
    - exercise in code with solutions"

Above is a starter prompt, see how you can use it and tweak it to your liking.

### History bot

Here's some prompts you could be using:

- "You are Abe Lincoln, tell me about yourself in 3 sentences, and respond using grammar and words like Abe would have used"
- "You are Abe Lincoln, respond using grammar and words like Abe would have used:

   Tell me about your greatest accomplishments, in 300 words:"

## Knowledge check

What does the concept temperature do?

1. It controls how random the output is.
1. It controls how big the response is.
1. It controls how many tokens are used.

A: 1

What's a good way to store secrets like API keys?

1. In code.
1. In a file.
1. In environment variables.

A: 3, because environment variables are not stored in code and can be loaded from the code. 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
